# IndyCar PDF scraper

In [ ]:
!pip install pypdf pdfplumber requests pandas

In [ ]:
import re
import numpy as np
import pandas as pd
import pdfplumber
import requests
import os
import time

from pathlib import Path
from datetime import timedelta

In [ ]:
def split_name(full_name: str) -> tuple:
    full_name = full_name.strip().replace(" (R)", "").strip()
    if "," in full_name:
        parts = full_name.split(",", 1)
        last  = parts[0].strip()
        first = parts[1].strip()
    else:
        parts = full_name.split()
        first = parts[0] if parts else ""
        last  = " ".join(parts[1:]) if len(parts) > 1 else ""
    return first, last

def extract_text(pdf_path: str) -> str:
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += (page.extract_text() or "") + "\n"
    return text

In [ ]:
# Race Result Parser - Most Important Data
def parse_race_result_file(text:str) -> dict:
  # A value the PDF doesn't show stays NaN, so a failed read can't pass for a real 0
  def find(pattern, cast):
    m = re.search(pattern, text)
    return cast(m.group(1)) if m else np.nan

  result = {
      "TotalCautionLaps": find(r"Caution Laps:\s+(\d+)", int),
      "TotalLeadChanges": find(r"Lead Changes:\s+(\d+)", int),
      "TotalRaceLaps": find(r"End of Lap\s+(\d+):", int),
      "TrackLength": find(r"([\d.]+)\s+mile\(s\)", float),
      "AvgRaceSpeed": find(r"Avg Speed:\s+([\d.]+)", float),
      "FastestLapSpeed": find(r"Fastest Lap:\s+([\d.]+)", float),
  }

  section = text[text.find("Caution Summary"):] if "Caution Summary" in text else ""
  caution = re.compile(
      r"(\d+)\s+to\s+(\d+)\s+(\d+)\s+([A-Z][^\n]+)"
  )

  cautions = []
  for m in caution.finditer(section):
    cautions.append({
        "start_lap": int(m.group(1)),
        "end_lap": int(m.group(2)),
        "duration": int(m.group(3)),
        "reason": m.group(4).split(":")[0].strip(),
    })

  result["TotalCautions"] = len(cautions) if section else np.nan
  result["Cautions"] = cautions

  return result

In [ ]:
# Pit Summary Parser - Important Data
def parse_pit_summary_file(text: str) -> list[dict]:
  pit_stops = []

  drivers = re.compile(
      r"(\d{1,2})\s+(\d{1,2})\s+(.+?)\s+(D/[CH]/F)\s+(\d+)\s+Pit Stops"
  )

  stops = re.compile(
      r"^(\d+)\s+(\d+)\s+(\d+)\s+(\d{2}:\d{2}:\d{2}\.\d+)$"
  )

  driver = None

  for line in text.split("\n"):
    dm = drivers.search(line)
    if dm:
      first, last = split_name(dm.group(3))
      driver = {
          "carNum": int(dm.group(2)),
          "FName":first,
          "LName": last,
          "TotalPitStops": int(dm.group(5)),
      }
      continue

    sm = stops.match(line.strip())
    if sm and driver:
      pit_stops.append({
          **driver,
          "StopNumber": int(sm.group(1)),
          "DriverLap": int(sm.group(2)),
          "RaceLap": int(sm.group(3)),
          "Time": sm.group(4),
      })

  return pit_stops

In [ ]:
# Lap Chart Parser
def parse_lap_chart_file(text: str) -> dict:
  drivers = re.compile(
      r"(\d{1,2})\s+-\s+(.+?)\s+\((\d+)\)\s+(\d+)\s+([\d\s]+)"
  )

  driver_info = {}
  positions = []

  for line in text.split("\n"):
    m = drivers.match(line)
    if m:
      car_num = int(m.group(1))
      first, last = split_name(m.group(2))
      position =  int(m.group(4))
      car_nums = [int(x) for x in m.group(5).strip().split()]

      if car_num not in driver_info:
        driver_info[car_num] = (first, last)

      positions.append((position, car_nums))

  # Find what position each car is at
  car_positions = {car_num: [] for car_num in driver_info}

  #rows by position
  from collections import defaultdict
  position_groups = defaultdict(list)
  for position, car_nums in positions:
    position_groups[position].append(car_nums)

  # Check all pages
  all_pages = {}
  for position, groups in position_groups.items():
    all_pages[position] = []
    for group in groups:
      all_pages[position].extend(group)


  total_laps = max(len(v) for v in all_pages.values())

  # Find positions per lap
  for lap_i in range(total_laps):
    for position, car_nums in all_pages.items():
      if lap_i < len(car_nums):
        car_num = car_nums[lap_i]
        if car_num in car_positions and len(car_positions[car_num]) == lap_i:
          car_positions[car_num].append(position)

  return {
      car_num: {
          "DriverName": driver_info[car_num],
          "Positions": car_positions[car_num],
      }
      for car_num in driver_info
  }

In [ ]:
# Section Results Parser
def parse_section_data(text: str) -> list[dict]:
    records = []
    current_car = None
    current_driver = None
    next_lap = None
    seen = set()

    for line in text.split("\n"):
        dm = re.match(r"Section Data for Car (\d+) - (.+)", line)
        if dm:
            current_car = int(dm.group(1))
            current_driver = dm.group(2).strip()
            continue

        lm = re.match(r"^(\d+)$", line.strip())
        if lm and current_car is not None:
            next_lap = int(lm.group(1))
            continue

        parts = line.strip().split()
        if parts and parts[0] == "T" and current_car is not None and next_lap is not None:
            try:
                vals = [float(x) for x in parts[1:]]
            except ValueError:
                next_lap = None
                continue
            n = len(vals)

            if n < 3:
                next_lap = None
                continue

            key = (current_car, next_lap)
            if key in seen:
                next_lap = None
                continue
            seen.add(key)

            # Detect: last value is much smaller than second-to-last
            if n >= 4 and vals[-1] < vals[-2] and vals[-1] < vals[-2] * 0.6:
                lap_time = vals[-2]
            else:
                lap_time = vals[-1]

            first, last = split_name(current_driver)
            records.append({
                "car": current_car,
                "FName": first,
                "LName": last,
                "lap": next_lap,
                "lap_time": lap_time,
            })
            next_lap = None

    return records

In [ ]:
def parse_qualy(text: str) -> list[dict]:
    results = []
    pattern = re.compile(
        r"(\d{1,2})\s+(\d{1,2})\s+(.+?)\s+(D/[CH]/F)\s+"
        r"(\d{2}:\d{2}\.\d+)\s+([\d.]+)\s+(.+)"
    )

    for line in text.split("\n"):
      m = pattern.match(line.strip())
      if m:
        first, last = split_name(m.group(3))
        results.append({
            "carNum": int(m.group(2)),
            "FName": first,
            "LName": last,
            "Rank": int(m.group(1)),
            "QualyTime": m.group(5),
            "QualySpeed": float(m.group(6)),
            "Session": m.group(7).strip(),
        })

    return results

def parse_practice(text: str) -> list[dict]:
    results = []
    pattern = re.compile(
        r"(\d{1,2})\s+(\d{1,2})\s+(.+?)\s+(D/[CH]/F)\s+"
        r"(?:Practice \d.*?\s+)?"
        r"(\d{2}:\d{2}\.\d+)\s+([\d.]+)\s+.*?(\d+)\s*$"
    )
    for line in text.split("\n"):
        m = pattern.match(line.strip())
        if m:
            first, last = split_name(m.group(3))
            results.append({
                "carNum": int(m.group(2)),
                "FName": first,
                "LName": last,
                "Rank": int(m.group(1)),
                "bestTime": m.group(5),
                "Speed": float(m.group(6)),
                "totalLaps": int(m.group(7)),
            })
    return results

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/indycar")
BASE_DATASET = DRIVE / "datasets" / "IndyCar_dataset_v25.csv"  # the results file you typed the latest race into
BASE_DIR = DRIVE / "races"              # one folder of PDFs per race, named <EventID>_<track>_<year>
OUTPUT_DIR = DRIVE / "scraped_data"     # the five CSVs the metrics generator reads

def folder_event_id(name):
    """INDYCAR event ID at the start of a race folder's name, e.g. 6463_the_thermal_club_2025."""
    prefix = name.split("_")[0]
    return int(prefix) if prefix.isdigit() else None

In [ ]:
## Dowloads report files of each race

df = pd.read_csv(BASE_DATASET)
df["EventDate"] = pd.to_datetime(df["EventDate"])

# Get one row per event

events = (df[["EventID", "Track", "EventDate"]]
          .drop_duplicates("EventID")
          .assign(EventDate=lambda x: pd.to_datetime(x["EventDate"]))
          .sort_values("EventDate")
          .reset_index(drop=True))

# PDFs exist from 2013 on; races already downloaded are skipped below
events = events[events["EventDate"].dt.year >= 2013]
valid_ids = set(events["EventID"])

In [ ]:
# One-time migration. Race folders used to be numbered by their position in the dataset
# (e.g. 207_the_thermal_club_2025), so fixing or adding a past race renumbered every folder
# after it. This renames them to start with INDYCAR's event ID and updates the scraped CSVs
# to match. Run it with DRY_RUN = True to see what would change, then with False.
# Running it again later does nothing: renamed folders no longer match the old names.
DRY_RUN = True

LEGACY_FOLDERS = {
    "005_streets_of_st_petersburg_2013": 2645,
    "006_barber_motorsports_park_2013": 2654,
    "007_streets_of_long_beach_2013": 2682,
    "008_streets_of_sao_paulo_2013": 2695,
    "009_indianapolis_motor_speedway_2013": 2704,
    "012_texas_motor_speedway_2013": 2739,
    "013_milwaukee_mile_2013": 2743,
    "014_iowa_speedway_2013": 2753,
    "015_pocono_raceway_2013": 2762,
    "016_streets_of_toronto_2013": 2788,
    "017_streets_of_toronto_2013": 2769,
    "018_mid-ohio_sports_car_course_2013": 2799,
    "019_sonoma_raceway_2013": 2827,
    "020_streets_of_baltimore_2013": 2835,
    "021_reliant_park_2013": 2864,
    "022_reliant_park_2013": 2869,
    "023_auto_club_speedway_2013": 2877,
    "024_streets_of_st_petersburg_2014": 2888,
    "025_streets_of_long_beach_2014": 2911,
    "026_barber_motorsports_park_2014": 2924,
    "027_indianapolis_motor_speedway_2014": 2943,
    "028_indianapolis_motor_speedway_2014": 2988,
    "029_the_raceway_at_belle_isle_park_2014": 2993,
    "030_the_raceway_at_belle_isle_park_2014": 2998,
    "031_texas_motor_speedway_2014": 3002,
    "032_reliant_park_2014": 3009,
    "033_reliant_park_2014": 3014,
    "034_pocono_raceway_2014": 3024,
    "035_iowa_speedway_2014": 3031,
    "036_streets_of_toronto_2014": 3041,
    "037_streets_of_toronto_2014": 3036,
    "038_mid-ohio_sports_car_course_2014": 3056,
    "039_milwaukee_mile_2014": 3082,
    "040_sonoma_raceway_2014": 3106,
    "041_auto_club_speedway_2014": 3113,
    "042_streets_of_st_petersburg_2015": 3149,
    "043_nola_motorsports_park_2015": 3173,
    "044_streets_of_long_beach_2015": 3192,
    "045_barber_motorsports_park_2015": 3205,
    "046_indianapolis_motor_speedway_2015": 3226,
    "047_indianapolis_motor_speedway_2015": 3236,
    "048_the_raceway_at_belle_isle_park_2015": 3276,
    "049_the_raceway_at_belle_isle_park_2015": 3282,
    "050_texas_motor_speedway_2015": 3285,
    "051_streets_of_toronto_2015": 3292,
    "052_auto_club_speedway_2015": 3315,
    "053_milwaukee_mile_2015": 3323,
    "054_iowa_speedway_2015": 3327,
    "055_mid-ohio_sports_car_course_2015": 3341,
    "056_pocono_raceway_2015": 3361,
    "057_sonoma_raceway_2015": 3367,
    "058_streets_of_st_petersburg_2016": 3409,
    "059_phoenix_raceway_2016": 3413,
    "060_streets_of_long_beach_2016": 3423,
    "061_barber_motorsports_park_2016": 3447,
    "062_indianapolis_motor_speedway_2016": 3471,
    "063_indianapolis_motor_speedway_2016": 3484,
    "064_the_raceway_at_belle_isle_park_2016": 3512,
    "065_the_raceway_at_belle_isle_park_2016": 3518,
    "066_road_america_2016": 3527,
    "067_iowa_speedway_2016": 3557,
    "068_streets_of_toronto_2016": 3563,
    "069_mid-ohio_sports_car_course_2016": 3600,
    "070_pocono_raceway_2016": 3608,
    "071_texas_motor_speedway_2016": 3521,
    "072_watkins_glen_international_2016": 3616,
    "073_sonoma_raceway_2016": 3645,
    "074_streets_of_st_petersburg_2017": 3651,
    "075_streets_of_long_beach_2017": 3678,
    "076_barber_motorsports_park_2017": 3686,
    "077_phoenix_raceway_2017": 3703,
    "078_indianapolis_motor_speedway_2017": 3707,
    "079_indianapolis_motor_speedway_2017": 3740,
    "080_the_raceway_at_belle_isle_park_2017": 3748,
    "081_the_raceway_at_belle_isle_park_2017": 3752,
    "082_texas_motor_speedway_2017": 3755,
    "083_road_america_2017": 3762,
    "084_iowa_speedway_2017": 3783,
    "085_streets_of_toronto_2017": 3796,
    "086_mid-ohio_sports_car_course_2017": 3815,
    "087_pocono_raceway_2017": 3838,
    "088_world_wide_technology_raceway_2017": 3842,
    "089_watkins_glen_international_2017": 3848,
    "090_sonoma_raceway_2017": 3857,
    "091_streets_of_st_petersburg_2018": 5230,
    "092_phoenix_raceway_2018": 5229,
    "093_streets_of_long_beach_2018": 5232,
    "094_barber_motorsports_park_2018": 5231,
    "095_indianapolis_motor_speedway_2018": 5226,
    "096_indianapolis_motor_speedway_2018": 5225,
    "097_the_raceway_at_belle_isle_park_2018": 5228,
    "098_the_raceway_at_belle_isle_park_2018": 5227,
    "099_texas_motor_speedway_2018": 5233,
    "100_road_america_2018": 5239,
    "101_iowa_speedway_2018": 5238,
    "102_streets_of_toronto_2018": 5241,
    "103_mid-ohio_sports_car_course_2018": 5240,
    "104_pocono_raceway_2018": 5235,
    "105_world_wide_technology_raceway_2018": 5234,
    "106_portland_international_raceway_2018": 5237,
    "107_sonoma_raceway_2018": 5236,
    "108_streets_of_st_petersburg_2019": 5463,
    "109_circuit_of_the_americas_2019": 5477,
    "110_barber_motorsports_park_2019": 5475,
    "111_streets_of_long_beach_2019": 5476,
    "112_indianapolis_motor_speedway_2019": 5474,
    "113_indianapolis_motor_speedway_2019": 5473,
    "114_the_raceway_at_belle_isle_park_2019": 5472,
    "115_the_raceway_at_belle_isle_park_2019": 5478,
    "116_texas_motor_speedway_2019": 5471,
    "117_road_america_2019": 5469,
    "118_streets_of_toronto_2019": 5467,
    "119_iowa_speedway_2019": 5468,
    "120_mid-ohio_sports_car_course_2019": 5466,
    "121_pocono_raceway_2019": 5465,
    "122_world_wide_technology_raceway_2019": 5464,
    "123_portland_international_raceway_2019": 5470,
    "124_weathertech_raceway_laguna_seca_2019": 5479,
    "125_texas_motor_speedway_2020": 5733,
    "126_indianapolis_motor_speedway_2020": 5741,
    "127_road_america_2020": 5742,
    "128_road_america_2020": 5748,
    "129_iowa_speedway_2020": 5751,
    "130_iowa_speedway_2020": 5754,
    "131_indianapolis_motor_speedway_2020": 5777,
    "132_world_wide_technology_raceway_2020": 5780,
    "133_world_wide_technology_raceway_2020": 5782,
    "134_mid-ohio_sports_car_course_2020": 5762,
    "135_mid-ohio_sports_car_course_2020": 5765,
    "136_indianapolis_motor_speedway_2020": 5783,
    "137_indianapolis_motor_speedway_2020": 5784,
    "138_streets_of_st_petersburg_2020": 5790,
    "139_barber_motorsports_park_2021": 5797,
    "140_streets_of_st_petersburg_2021": 5798,
    "141_texas_motor_speedway_2021": 5810,
    "142_texas_motor_speedway_2021": 5809,
    "143_indianapolis_motor_speedway_2021": 5808,
    "144_indianapolis_motor_speedway_2021": 5807,
    "145_the_raceway_at_belle_isle_park_2021": 5806,
    "146_the_raceway_at_belle_isle_park_2021": 5812,
    "147_road_america_2021": 5813,
    "148_mid-ohio_sports_car_course_2021": 5801,
    "149_streets_of_nashville_2021": 5800,
    "150_indianapolis_motor_speedway_2021": 5802,
    "151_world_wide_technology_raceway_2021": 5799,
    "152_portland_international_raceway_2021": 5805,
    "153_weathertech_raceway_laguna_seca_2021": 5803,
    "154_streets_of_long_beach_2021": 5811,
    "155_streets_of_st_petersburg_2022": 5921,
    "156_texas_motor_speedway_2022": 5912,
    "157_streets_of_long_beach_2022": 5906,
    "158_barber_motorsports_park_2022": 5907,
    "160_indianapolis_motor_speedway_2022": 5909,
    "161_the_raceway_at_belle_isle_park_2022": 5910,
    "162_road_america_2022": 5911,
    "163_mid-ohio_sports_car_course_2022": 5913,
    "164_streets_of_toronto_2022": 5920,
    "165_iowa_speedway_2022": 5914,
    "166_iowa_speedway_2022": 5915,
    "167_indianapolis_motor_speedway_2022": 5916,
    "168_streets_of_nashville_2022": 5917,
    "169_world_wide_technology_raceway_2022": 5918,
    "170_portland_international_raceway_2022": 5919,
    "171_weathertech_raceway_laguna_seca_2022": 5905,
    "172_streets_of_st_petersburg_2023": 6125,
    "173_texas_motor_speedway_2023": 6139,
    "174_streets_of_long_beach_2023": 6138,
    "175_barber_motorsports_park_2023": 6137,
    "176_indianapolis_motor_speedway_2023": 6136,
    "177_indianapolis_motor_speedway_2023": 6135,
    "178_streets_of_detroit_2023": 6134,
    "179_road_america_2023": 6140,
    "180_mid-ohio_sports_car_course_2023": 6133,
    "181_streets_of_toronto_2023": 6131,
    "182_iowa_speedway_2023": 6130,
    "183_iowa_speedway_2023": 6129,
    "184_streets_of_nashville_2023": 6127,
    "185_indianapolis_motor_speedway_2023": 6128,
    "186_world_wide_technology_raceway_2023": 6126,
    "187_portland_international_raceway_2023": 6132,
    "188_weathertech_raceway_laguna_seca_2023": 6141,
    "189_streets_of_st_petersburg_2024": 6298,
    "190_streets_of_long_beach_2024": 6297,
    "191_barber_motorsports_park_2024": 6312,
    "192_indianapolis_motor_speedway_2024": 6301,
    "193_indianapolis_motor_speedway_2024": 6309,
    "194_streets_of_detroit_2024": 6296,
    "195_road_america_2024": 6308,
    "196_weathertech_raceway_laguna_seca_2024": 6311,
    "197_mid-ohio_sports_car_course_2024": 6303,
    "198_iowa_speedway_2024": 6310,
    "199_iowa_speedway_2024": 6307,
    "200_streets_of_toronto_2024": 6306,
    "201_world_wide_technology_raceway_2024": 6302,
    "202_portland_international_raceway_2024": 6305,
    "203_milwaukee_mile_2024": 6299,
    "204_milwaukee_mile_2024": 6313,
    "205_nashville_superspeedway_2024": 6304,
    "206_streets_of_st_petersburg_2025": 4444,
    "207_the_thermal_club_2025": 6463,
    "208_streets_of_long_beach_2025": 6457,
    "209_barber_motorsports_park_2025": 6458,
    "210_indianapolis_motor_speedway_2025": 6459,
    "211_indianapolis_motor_speedway_2025": 6460,
    "212_streets_of_detroit_2025": 6461,
    "213_world_wide_technology_raceway_2025": 6462,
    "214_mid-ohio_sports_car_course_2025": 6464,
    "215_iowa_speedway_2025": 6465,
    "216_iowa_speedway_2025": 6466,
    "217_streets_of_toronto_2025": 6467,
    "218_weathertech_raceway_laguna_seca_2025": 6468,
    "219_portland_international_raceway_2025": 6469,
    "220_milwaukee_mile_2025": 6470,
    "221_nashville_superspeedway_2025": 6471,
    "222_streets_of_st_petersburg_2026": 6472,
    "223_phoenix_raceway_2026": 6473,
    "224_streets_of_arlington_2026": 6474,
    "225_barber_motorsports_park_2026": 6475,
    "226_streets_of_long_beach_2026": 6476,
    "227_indianapolis_motor_speedway_2026": 6744,
    "228_indianapolis_motor_speedway_2026": 6743,
    "229_streets_of_detroit_2026": 6742,
    "230_world_wide_technology_raceway_2026": 6741,
    "231_road_america_2026": 6747,
    "232_mid-ohio_sports_car_course_2026": 6740,
    "233_nashville_superspeedway_2026": 6748,
    "234_portland_international_raceway_2026": 6733,
    "235_streets_of_markham_2026": 6737,
    "236_streets_of_washington_2026": 6796,
    "237_milwaukee_mile_2026": 6739,
    "238_milwaukee_mile_2026": 6738,
}

renames, unmatched = [], []
for folder in sorted(p for p in BASE_DIR.iterdir() if p.is_dir() and not p.name.startswith("_")):
    if folder.name in LEGACY_FOLDERS:
        new_name = f"{LEGACY_FOLDERS[folder.name]}_{folder.name.split('_', 1)[1]}"
        renames.append((folder, BASE_DIR / new_name))
    elif folder_event_id(folder.name) not in valid_ids:
        unmatched.append(folder)

print(f"{len(renames)} folders to rename; {len(unmatched)} match no race in the results file")
for old, new in renames[:5]:
    print(f"  {old.name} -> {new.name}")
for folder in unmatched:
    print(f"  no matching race, will move to _unmatched/: {folder.name}")

if not DRY_RUN:
    for old, new in renames:
        old.rename(new)
    if unmatched:
        (BASE_DIR / "_unmatched").mkdir(exist_ok=True)
        for folder in unmatched:
            folder.rename(BASE_DIR / "_unmatched" / folder.name)
    for name in ["race_metadata", "cautions", "pit_stops", "lap_positions", "lap_times"]:
        path = OUTPUT_DIR / f"{name}.csv"
        if path.exists():
            scraped = pd.read_csv(path)
            scraped["event_id"] = scraped["event_id"].replace(LEGACY_FOLDERS)
            # rows from folders that match no race were never used by the metrics generator
            scraped = scraped[pd.to_numeric(scraped["event_id"], errors="coerce").notna()]
            scraped.to_csv(path, index=False)
    print("✅ Migration done")

In [ ]:
BASE_URL = "http://www.imscdn.com/INDYCAR/Documents"
BASE_DIR.mkdir(exist_ok=True)

def get_filenames(year):
    results_file = "indycar-race-lapfinal.pdf" if year <= 2014 else "indycar-race-results.pdf"
    return {
        "results": [results_file],
        "event_summary": ["indycar-eventsummary.pdf"],
        "pit_stops": ["indycar-race-pitstopsummary.pdf"],
        "lap_chart": ["indycar-race-lapchart.pdf"],
        "section_data": ["indycar-sectionresults-race.pdf"],
    }

def download_pdf(url, dest_path):
    """Download a single PDF with error handling."""
    try:
        response = requests.get(url, timeout=30)
        if response.status_code == 200:
            with open(dest_path, "wb") as f:
                f.write(response.content)
            return True
        else:
            print(f"  ❌ {response.status_code} — {url}")
            return False
    except Exception as e:
        print(f"  ❌ Error: {e}")
        return False

def get_date_candidates(race_date):
    return [
        race_date,
        race_date - timedelta(days=1),
        race_date - timedelta(days=2),
    ]
# Folders still named the old way would be downloaded again under new names
old_style = [f.name for f in BASE_DIR.iterdir()
             if f.is_dir() and not f.name.startswith("_") and folder_event_id(f.name) not in valid_ids]
if old_style:
    raise RuntimeError(f"Run the one-time migration cell first: {len(old_style)} folders use old names, e.g. {old_style[0]}")

# --- Main download loop ---
total_downloaded = 0
total_failed     = 0

for i, row in events.iterrows():
    event_id   = row["EventID"]
    event_date = row["EventDate"]
    track      = row["Track"]
    year       = event_date.year

    # Build folder name matching our structure
    track_clean = (track
                   .replace(" ", "_")
                   .replace(".", "")
                   .replace("/", "_")
                   .replace("(", "")
                   .replace(")", "")
                   .lower())
    folder_name = f"{event_id}_{track_clean}_{year}"
    folder_path = BASE_DIR / folder_name
    folder_path.mkdir(exist_ok=True)

    date_str   = event_date.strftime("%Y-%m-%d")
    filenames  = get_filenames(year)

    print(f"\n📥 {folder_name}")

    for doc_type, candidates in filenames.items():
      downloaded = False
      date_candidates = get_date_candidates(event_date)

      for filename in candidates:
        if downloaded:
          break
        dest_path = folder_path / filename

        # Skip if already downloaded
        if dest_path.exists():
            print(f"  ✓ Already exists — {filename}")
            downloaded = True
            break

        for try_date in date_candidates:
          date_str = try_date.strftime("%Y-%m-%d")
          url = f"{BASE_URL}/{event_id}/{date_str}/{filename}"
          success = download_pdf(url, dest_path)

          if success:
              print(f"  ✅ {filename}")
              total_downloaded += 1
              downloaded = True
              break

          time.sleep(1)

      if not downloaded:
        print(f"  ❌ No file found for {doc_type}")
        total_failed += 1

    time.sleep(3)  # be polite to the server

print(f"\n✅ Done! Downloaded: {total_downloaded} | Failed: {total_failed}")

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)

# Load existing progress if restarting
if (OUTPUT_DIR / "race_metadata.csv").exists():
    all_race_metadata = pd.read_csv(OUTPUT_DIR / "race_metadata.csv").to_dict("records")
    all_cautions = pd.read_csv(OUTPUT_DIR / "cautions.csv").to_dict("records")
    all_pit_stops = pd.read_csv(OUTPUT_DIR / "pit_stops.csv").to_dict("records")
    all_lap_positions = pd.read_csv(OUTPUT_DIR / "lap_positions.csv").to_dict("records")
    all_lap_times = pd.read_csv(OUTPUT_DIR / "lap_times.csv").to_dict("records")
    processed = {r["event_id"] for r in all_race_metadata}
    print(f"Resuming — {len(processed)} races already done")
else:
    all_race_metadata = []
    all_cautions = []
    all_pit_stops = []
    all_lap_positions = []
    all_lap_times = []
    processed = set()

folders = sorted([f for f in BASE_DIR.iterdir() if f.is_dir() and not f.name.startswith("_")])

for i, folder in enumerate(folders):
    event_id = folder_event_id(folder.name)
    if event_id not in valid_ids:
        print(f"  ⚠️ Skipping {folder.name}: no race with that EventID in the results file")
        continue

    if event_id in processed:
        continue

    print(f"[{i+1}/{len(folders)}] Processing {event_id}...")

    # --- Race Results ---
    race_results_path = folder / "indycar-race-results.pdf"
    if not race_results_path.exists():
        race_results_path = folder / "indycar-race-lapfinal.pdf"
    if not race_results_path.exists():
        matches = list(folder.glob("indycar-race-results*.pdf")) or list(folder.glob("indycar-race-lapfinal*.pdf"))
        race_results_path = matches[0] if matches else race_results_path
    if race_results_path.exists():
        try:
            text = extract_text(str(race_results_path))
            data = parse_race_result_file(text)
            cautions = data.pop("Cautions", [])
            missing = [k for k, v in data.items() if pd.isna(v)]
            if missing:
                print(f"  ⚠️ Not found in the race results PDF: {', '.join(missing)}")
            data["event_id"] = event_id
            all_race_metadata.append(data)
            for c in cautions:
                c["event_id"] = event_id
                all_cautions.append(c)
        except Exception as e:
            print(f"  ❌ Race results failed: {e}")

    # --- Pit Stop Summary ---
    pit_path = folder / "indycar-race-pitstopsummary.pdf"
    if not pit_path.exists():
        matches = list(folder.glob("indycar-race-pitstopsummary*.pdf"))
        pit_path = matches[0] if matches else pit_path
    if pit_path.exists():
        try:
            text = extract_text(str(pit_path))
            stops = parse_pit_summary_file(text)
            for s in stops:
                s["event_id"] = event_id
                all_pit_stops.append(s)
        except Exception as e:
            print(f"  ❌ Pit stops failed: {e}")

    # --- Lap Chart ---
    lap_chart_path = folder / "indycar-race-lapchart.pdf"
    if not lap_chart_path.exists():
        matches = list(folder.glob("indycar-race-lapchart*.pdf"))
        lap_chart_path = matches[0] if matches else lap_chart_path
    if lap_chart_path.exists():
        try:
            text = extract_text(str(lap_chart_path))
            data = parse_lap_chart_file(text)
            for car, info in data.items():
                first, last = info["DriverName"]
                for lap_idx, position in enumerate(info["Positions"], start=1):
                    all_lap_positions.append({
                        "event_id": event_id,
                        "car": car,
                        "FName": first,
                        "LName": last,
                        "lap": lap_idx,
                        "position": position,
                    })
        except Exception as e:
            print(f"  ❌ Lap chart failed: {e}")

    # --- Section Data (Lap Times) ---
    section_path = folder / "indycar-sectionresults-race.pdf"
    if not section_path.exists():
        matches = list(folder.glob("indycar-sectionresults-race*.pdf"))
        section_path = matches[0] if matches else section_path
    if section_path.exists():
        try:
            print(f"  Parsing section data...")
            text = extract_text(str(section_path))
            records = parse_section_data(text)
            for r in records:
                r["event_id"] = event_id
                all_lap_times.append(r)
        except Exception as e:
            print(f"  ❌ Section data failed: {e}")

    # Save all 5 CSVs after every race
    pd.DataFrame(all_race_metadata).to_csv(OUTPUT_DIR / "race_metadata.csv", index=False)
    pd.DataFrame(all_cautions).to_csv(OUTPUT_DIR / "cautions.csv", index=False)
    pd.DataFrame(all_pit_stops).to_csv(OUTPUT_DIR / "pit_stops.csv", index=False)
    pd.DataFrame(all_lap_positions).to_csv(OUTPUT_DIR / "lap_positions.csv", index=False)
    pd.DataFrame(all_lap_times).to_csv(OUTPUT_DIR / "lap_times.csv", index=False)

print(f"\n✅ Done!")
print(f"  race_metadata: {len(all_race_metadata)} races")
print(f"  cautions: {len(all_cautions)} caution periods")
print(f"  pit_stops: {len(all_pit_stops)} pit stop records")
print(f"  lap_positions: {len(all_lap_positions)} position records")
print(f"  lap_times: {len(all_lap_times)} lap time records")

In [ ]:
# A race whose results PDF failed is processed again on the next run, so drop repeated rows in every file
for name in ["race_metadata", "cautions", "pit_stops", "lap_positions", "lap_times"]:
    path = OUTPUT_DIR / f"{name}.csv"
    pd.read_csv(path).drop_duplicates().to_csv(path, index=False)

In [ ]:
for f in OUTPUT_DIR.glob("*.csv"):
    df = pd.read_csv(f)
    print(f"{f.name}: {len(df)} rows, {len(df.columns)} cols")